# HomeReach Melbourne — 12: A damped rent-change correction

**One fixed follow-up candidate, not a search for a winning score.** This notebook learns a quarterly rent **change**, applies half of that estimate, and adds it to the previous quarter's rent:

`forecast = last observed rent + 0.5 × predicted change`

The candidate was designed **after** inspecting the earlier validation errors. Its settings are fixed before this candidate is evaluated, but this is still **development on an inspected validation set**, not independent confirmation, preregistration, or a real-time backtest.

Use the existing **homereach** kernel on **feat/rental-forecast-baselines**. Run from top to bottom. Earlier notebook variables are not required; their saved outputs are.

The notebook adds **one** candidate to the three previously saved methods, retains every one of the 589 validation targets, and fits no models on the reserved final period. It does not tune the damping weight, apply an oracle change-direction rule, clip/round predictions, or alter the live explorer. Current and lagged published counts and transport data are excluded.

**Inputs:** the audited rental panel and the completed notebooks 09–11.  
**Writes only:** `data/interim/rental_correction_validation/` and `docs/rental_correction_validation.md`.  
**No deployment, final-period evaluation, model pickle or current-rent forecast.**

## 1. Verify the rental input and research branch

The source CSV is verified against its own local pipeline report. Its full structure is checked, then the final-period values are discarded before modelling. No baseline, app or source file is overwritten. Date comparisons explicitly use the same resolution on both sides.

In [1]:
print("Starting notebook 12 — damped rent-change validation...", flush=True)
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import math
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly
from IPython.display import display
from homereach.io.dffh_rent import COLUMN_ORDER, read_panel_csv


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    require(isinstance(value, dict), f"Expected a JSON object: {path}")
    return value


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pyproject.toml").is_file() and (p / "data").is_dir()), None)
require(ROOT is not None, "Open this notebook from the existing HomeReach project.")
branch_result = subprocess.run(
    ["git", "-C", str(ROOT), "branch", "--show-current"],
    capture_output=True, text=True, timeout=10, check=True,
)
BRANCH = branch_result.stdout.strip()
require(BRANCH == "feat/rental-forecast-baselines",
        "Use the research branch feat/rental-forecast-baselines before running notebook 12; do not develop on deployed main.")

INPUT = ROOT / "data/processed/rent_quarterly.csv"
QUALITY_PATH = ROOT / "data/processed/rent_quality_report.json"
SPEC_PATH = ROOT / "config/rent_snapshot.json"
PROTOCOL_PATH = ROOT / "config/rental_forecast_protocol.json"
for path in [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]:
    require(path.is_file(), f"Missing required file: {path.relative_to(ROOT)}")

spec = read_json(SPEC_PATH)
quality_report = read_json(QUALITY_PATH)
protocol = read_json(PROTOCOL_PATH)
EXPECTED_SOURCE_SHA256 = "b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837"
require(spec["sha256"] == protocol["source_sha256"] == EXPECTED_SOURCE_SHA256,
        "This notebook requires the audited 2025Q3 rental snapshot. Do not change fingerprints to bypass it.")
protected_paths = [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]
raw_rent = ROOT / "data/raw/rent" / spec["filename"]
if raw_rent.is_file():
    require(sha256_file(raw_rent) == EXPECTED_SOURCE_SHA256, "The original rental workbook changed.")
    protected_paths.append(raw_rent)
for relative in ["data/processed/rent_quarterly.parquet", "docs/data_registry.csv", "README.md"]:
    if (ROOT / relative).is_file():
        protected_paths.append(ROOT / relative)
for directory in ["app", "deploy"]:
    protected_paths.extend(p for p in (ROOT / directory).rglob("*")
                           if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc")
input_hashes = {p: sha256_file(p) for p in sorted(set(protected_paths))}
print("Research branch:", BRANCH)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Protected existing files:", len(input_hashes))

require(quality_report.get("source", {}).get("sha256") == EXPECTED_SOURCE_SHA256,
        "Rental quality report belongs to another workbook.")
require(quality_report.get("output_sha256", {}).get(INPUT.name) == input_hashes[INPUT],
        "Rental CSV differs from its pipeline report. Investigate; do not edit the hash.")
require(quality_report.get("csv_readback_matches") is True,
        "The rental pipeline has not recorded a successful CSV read-back.")
q = quality_report.get("quality", {})
require(q.get("rows") == 3286 and q.get("lga_count") == 31 and q.get("source_values_verified") == 6572,
        "Unexpected rental quality checkpoint.")
panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER, "The 14-column rental schema changed.")
require(len(panel) == 3286 and panel.rental_area_name.nunique() == 31, "Unexpected panel size.")
require(not panel.duplicated(["rental_area_name", "property_type", "quarter"]).any(), "Duplicate rental key.")
for column, expected in {
    "property_type": "2 bedroom flats", "area_type": "LGA", "source_file": spec["filename"],
    "source_sheet": spec["sheet"], "source_reporting_quarter": "2025Q3",
}.items():
    require(panel[column].eq(expected).all(), f"Unexpected source scope: {column}")
for column in ["median_weekly_rent", "published_count"]:
    require(panel[column].notna().all() and np.isfinite(panel[column].astype(float)).all()
            and panel[column].gt(0).all(), f"Invalid numeric input: {column}")
periods = pd.PeriodIndex(panel.quarter, freq="Q-DEC")
expected_dates = periods.to_timestamp(how="end").normalize().as_unit("ns")
actual_dates = pd.DatetimeIndex(panel.quarter_end).as_unit("ns")
require(np.array_equal(actual_dates.asi8, expected_dates.asi8), "Quarter labels and dates disagree.")
ALL_QUARTERS = pd.period_range("1999Q2", "2025Q3", freq="Q-DEC")
for _, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(sorted(group.quarter.tolist()) == ALL_QUARTERS.astype(str).tolist(), "Missing or altered quarters.")
require(panel.source_median_cell.str.fullmatch(r"[A-Z]+[1-9][0-9]*").all(), "Missing source cell references.")
print("RENTAL INPUT CHECKPOINT PASSED")
print("Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106")
print("Source CSV matches its local pipeline fingerprint: True")
print("Transport data loaded: False | Holdout forecasts scored: 0")
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
print("scikit-learn:",sklearn.__version__)


Starting notebook 12 — damped rent-change validation...
Research branch: feat/rental-forecast-baselines
Python: 3.12.14 | pandas: 3.0.3
Protected existing files: 26
RENTAL INPUT CHECKPOINT PASSED
Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106
Source CSV matches its local pipeline fingerprint: True
Transport data loaded: False | Holdout forecasts scored: 0
scikit-learn: 1.8.0


## 2. Freeze one candidate and verify the earlier evidence

The original baseline/Ridge/diagnostic protocols are unchanged. This separate specification uses:

| Item | Fixed decision |
|---|---|
| Learning target | `rent(t) − rent(t−1)` |
| Historical inputs | `rent(t−1)−rent(t−2)`, `rent(t−2)−rent(t−3)`, `rent(t−3)−rent(t−4)` |
| Calendar inputs | Known target-quarter Q2/Q3/Q4 flags; Q1 reference |
| Estimator | Separate LGA models; StandardScaler → Ridge, alpha=1, intercept, SVD solver |
| Damping | **0.5**, fixed halfway correction toward last observation |
| Training | Expanding earlier targets; first eligible target 2000Q2 |
| Validation | The same 2019Q1–2023Q3 targets |
| Reserved final period | 2023Q4–2025Q3, not forecast or scored |

A weight of 0.5 is a transparent **heuristic**, not a fitted probability or an optimal weight. It halves the estimated adjustment; it does not guarantee a zero adjustment for unchanged medians, nor impose a dollar cap.

This changes the target, historical features and final adjustment together. We cannot attribute an accuracy difference to any one of those components alone. No alternative parameter combinations are scored here.

In [2]:
# The original protocol is immutable; this is a separate model specification.
RIDGE_PROTOCOL_PATH = ROOT / 'config/rental_ridge_protocol.json'
BASE_DIR = ROOT / 'data/interim/rental_forecast_baselines'
BASE_MANIFEST_PATH = BASE_DIR / 'baseline_manifest.json'
for p in [RIDGE_PROTOCOL_PATH, BASE_MANIFEST_PATH]:
    require(p.is_file(), f'Missing required input: {p.relative_to(ROOT)}')
ridge_protocol = read_json(RIDGE_PROTOCOL_PATH)
baseline_manifest = read_json(BASE_MANIFEST_PATH)

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

BASE_PROTOCOL_HASH = canonical_hash(protocol)
require(BASE_PROTOCOL_HASH == ridge_protocol['baseline_protocol_sha256'],
        'The baseline protocol differs from the reviewed design; do not edit it to bypass this check.')
require(baseline_manifest['protocol_sha256'] == BASE_PROTOCOL_HASH,
        'Notebook 09 outputs use a different protocol.')
require(baseline_manifest['input_csv_sha256'] == input_hashes[INPUT],
        'Notebook 09 outputs use another rental CSV. Investigate the change first.')
require(baseline_manifest['source_workbook_sha256'] == EXPECTED_SOURCE_SHA256,
        'Notebook 09 source vintage changed.')
bq = baseline_manifest['quality']
require(bq['validation_targets_per_model'] == 589 and bq['holdout_forecasts'] == 0
        and bq['holdout_scores'] == 0 and bq['csv_readback_matches'] is True,
        'Notebook 09 is incomplete or the reserved final period has already been scored.')
for key in ['source_sha256','target','property_type','first_quarter','last_quarter',
            'validation_start','validation_end','holdout_start','holdout_end','forecast_horizon_quarters']:
    require(ridge_protocol[key] == protocol[key], f'Experiment/baseline scope differs: {key}')
FEATURES = ['rent_lag_1','rent_lag_2','rent_lag_3','rent_lag_4',
            'target_is_q2','target_is_q3','target_is_q4']
LAGS = [1, 2, 3, 4]
RIDGE_NAME = 'ridge_lags_seasonal_alpha1'
require(ridge_protocol['numeric_features'] == FEATURES and ridge_protocol['lag_quarters'] == LAGS,
        'Feature specification changed; retain this experiment and create a new reviewed protocol instead.')
require(ridge_protocol['estimator'] == {'class':'Ridge','alpha':1.0,'fit_intercept':True,'solver':'svd'},
        'Unreviewed Ridge settings.')
require(ridge_protocol['model_name'] == RIDGE_NAME and ridge_protocol['minimum_training_examples'] == 40,
        'Unreviewed model specification.')
RIDGE_HASH = canonical_hash(ridge_protocol)
require(RIDGE_HASH == 'e98c4793b0e3c4f9bdfc0bfad9e54a9e1507fcc3851e49d8f1c21ec2672475e8',
        'The fixed Ridge protocol changed. Preserve this experiment before defining a new one.')
VALIDATION = pd.period_range(protocol['validation_start'], protocol['validation_end'], freq='Q-DEC')
HOLDOUT = pd.period_range(protocol['holdout_start'], protocol['holdout_end'], freq='Q-DEC')
require(len(VALIDATION) == 19 and len(HOLDOUT) == 8 and VALIDATION[-1] + 1 == HOLDOUT[0],
        'Chronological split changed.')
SERIES_KEYS = ['rental_area_name','property_type']
TARGET_KEYS = SERIES_KEYS + ['target_quarter']
MODEL_COLUMNS = SERIES_KEYS + ['quarter','median_weekly_rent','source_file','source_sheet','source_median_cell']

def development_only(frame):
    periods = pd.PeriodIndex(frame['quarter'], freq='Q-DEC')
    return frame.loc[periods <= VALIDATION[-1], MODEL_COLUMNS].copy()

development = development_only(panel)
require(len(development) == 3038, 'Unexpected development history (31 x 98 quarters).')
LGAS = sorted(development['rental_area_name'].unique())
del panel  # Discard the full table before creating features or fitting a model.

# Read existing baseline results using THEIR local manifest, not portable hashes.
BASE_FILES = ['validation_predictions.csv', 'validation_metrics_overall.csv',
              'validation_metrics_by_lga.csv','validation_metrics_by_quarter.csv']
baseline_tables = {}
for filename in BASE_FILES:
    p = BASE_DIR / filename
    require(p.is_file() and sha256_file(p) == baseline_manifest['output_sha256'][filename],
            f'Baseline output missing/changed: {filename}. Do not bypass the manifest.')
    baseline_tables[filename] = pd.read_csv(p, dtype=baseline_manifest['output_schemas'][filename],
                                          keep_default_na=False)
baseline_predictions = baseline_tables['validation_predictions.csv']
require(len(baseline_predictions) == 1178 and
        not baseline_predictions.duplicated(['model', *TARGET_KEYS]).any(), 'Baseline keys changed.')
require(set(baseline_predictions['model']) == {'last_observation','seasonal_naive'}
        and baseline_predictions['target_quarter'].isin(VALIDATION.astype(str)).all(),
        'Unexpected baseline method or forecast period.')

# Protect earlier notebooks, their evidence and the published app. Only our new outputs may be written.
more_protected = [RIDGE_PROTOCOL_PATH, BASE_MANIFEST_PATH]
more_protected += [p for p in BASE_DIR.rglob('*') if p.is_file()]
more_protected += [p for p in (ROOT/'notebooks').glob('*.ipynb') if not p.name.startswith('12_')]
more_protected += [p for p in (ROOT/'src').rglob('*.py')]
more_protected += [ROOT/'docs/rental_forecast_baselines.md']
for p in more_protected:
    if p.is_file(): input_hashes[p] = sha256_file(p)

OLD_RIDGE_NAME = RIDGE_NAME
OLD_RIDGE_DIR = ROOT / 'data/interim/rental_ridge_validation'
OLD_RIDGE_MANIFEST = OLD_RIDGE_DIR / 'ridge_manifest.json'
DIAG_DIR = ROOT / 'data/interim/rental_ridge_diagnostics'
DIAG_MANIFEST = DIAG_DIR / 'diagnostic_manifest.json'
DIAG_PROTOCOL_PATH = ROOT / 'config/rental_ridge_diagnostic_protocol.json'
CORRECTION_PATH = ROOT / 'config/rental_correction_protocol.json'
for p in [OLD_RIDGE_MANIFEST, DIAG_MANIFEST, DIAG_PROTOCOL_PATH, CORRECTION_PATH]:
    require(p.is_file(), f'Missing {p.relative_to(ROOT)}. Complete notebooks 09–11 first.')
rm, dm = read_json(OLD_RIDGE_MANIFEST), read_json(DIAG_MANIFEST)
correction_plan = read_json(CORRECTION_PATH)
CORRECTION_HASH = canonical_hash(correction_plan)
require(CORRECTION_HASH == '16f85a6c887eb12fb623c59259592791f620aa4a2c635f156fac9f5f080f6542',
        'The fixed candidate specification changed. Preserve this experiment; do not retune it silently.')
require(rm['input_csv_sha256'] == input_hashes[INPUT] and
        rm['baseline_manifest_sha256'] == sha256_file(BASE_MANIFEST_PATH), 'Prior Ridge evidence uses changed inputs.')
require(rm['ridge_protocol_sha256'] == RIDGE_HASH and rm['baseline_protocol_sha256'] == BASE_PROTOCOL_HASH,
        'Earlier Ridge protocol mismatch.')
require(dm['ridge_manifest_sha256'] == sha256_file(OLD_RIDGE_MANIFEST) and
        dm['baseline_manifest_sha256'] == sha256_file(BASE_MANIFEST_PATH), 'Diagnostic inputs have changed.')
require(dm['protocol_sha256'] == canonical_hash(read_json(DIAG_PROTOCOL_PATH)), 'Diagnostic protocol mismatch.')
require(rm['quality']['final_period_forecasts'] == rm['quality']['final_period_scores'] == 0,
        'Earlier final-period evaluations are not reserved.')
require(dm['quality']['holdout_forecasts_scored'] == 0 and dm['quality']['targets_removed'] == 0
        and dm['quality']['paired_targets'] == 589 and dm['quality']['csv_readback_matches'] is True,
        'The diagnostic checkpoint is incomplete or the scope changed.')
for name in ['source_sha256','property_type','validation_start','validation_end','holdout_start','holdout_end',
             'first_quarter','last_quarter','forecast_horizon_quarters']:
    require(correction_plan[name] == protocol[name], f'Candidate scope differs: {name}')


def read_verified_csv(directory, manifest, name):
    require(Path(name).name == name, 'Expected a plain evidence filename.')
    p = directory/name
    require(p.is_file() and sha256_file(p) == manifest['output_sha256'][name],
            f'Saved evidence missing or changed: {name}. Do not bypass its manifest.')
    f = pd.read_csv(p, dtype=manifest['output_schemas'][name], keep_default_na=False)
    require(f.columns.tolist() == list(manifest['output_schemas'][name]), f'Changed evidence schema: {name}')
    return f


old_tables = {name:read_verified_csv(OLD_RIDGE_DIR, rm, name) for name in [
    'validation_predictions_all_methods.csv','validation_metrics_overall.csv',
    'validation_metrics_by_lga.csv','validation_metrics_by_quarter.csv']}
old_predictions = old_tables['validation_predictions_all_methods.csv']
diag_summary = read_verified_csv(DIAG_DIR, dm, 'diagnostic_summary.csv')
require(len(old_predictions) == 1767 and
        set(old_predictions.model) == {'last_observation','seasonal_naive',OLD_RIDGE_NAME}
        and old_predictions.target_quarter.isin(VALIDATION.astype(str)).all()
        and not old_predictions.duplicated(['model',*TARGET_KEYS]).any(), 'Earlier forecast coverage changed.')
FEATURES = ['change_lag_1','change_lag_2','change_lag_3','target_is_q2','target_is_q3','target_is_q4']
RAW_FEATURES = [f'rent_lag_{k}' for k in LAGS] + [f'target_is_q{k}' for k in [2,3,4]]
MODEL_NAME = 'damped_change_ridge_alpha1_weight05'
ALPHA = 1.0
DAMPING = 0.5
require(correction_plan['numeric_features'] == FEATURES and correction_plan['model_name'] == MODEL_NAME
        and correction_plan['damping_weight'] == DAMPING, 'Changed candidate settings.')
for directory in [OLD_RIDGE_DIR, DIAG_DIR, ROOT/'config', ROOT/'src', ROOT/'scripts', ROOT/'tests']:
    for p in directory.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc':
            input_hashes[p] = sha256_file(p)
for p in [OLD_RIDGE_MANIFEST, DIAG_MANIFEST, DIAG_PROTOCOL_PATH, CORRECTION_PATH,
          ROOT/'docs/rental_ridge_validation.md', ROOT/'docs/rental_ridge_diagnostics.md']:
    if p.is_file(): input_hashes[p] = sha256_file(p)
OUT = ROOT / 'data/interim/rental_correction_validation'
MANIFEST_PATH = OUT / 'correction_manifest.json'
if MANIFEST_PATH.is_file():
    previous = read_json(MANIFEST_PATH)
    require(previous['protocol_sha256'] == CORRECTION_HASH and
            previous['input_csv_sha256'] == input_hashes[INPUT] and
            previous['ridge_manifest_sha256'] == input_hashes[OLD_RIDGE_MANIFEST] and
            previous['diagnostic_manifest_sha256'] == input_hashes[DIAG_MANIFEST],
            'An existing candidate run uses different evidence; preserve it for review.')
print('CORRECTION INPUT CHECKPOINT PASSED')
print('Earlier scored forecasts verified by manifest: 1767')
print('Development rows: 3038 | Reserved final-period feature rows: 0')
print('Fixed candidate: predict quarterly change; alpha=1; apply one half of the predicted adjustment')
print('Parameter combinations searched: 0 | Live release changes: 0')


CORRECTION INPUT CHECKPOINT PASSED
Earlier scored forecasts verified by manifest: 1767
Development rows: 3038 | Reserved final-period feature rows: 0
Fixed candidate: predict quarterly change; alpha=1; apply one half of the predicted adjustment
Parameter combinations searched: 0 | Live release changes: 0


## 3. Define causal features and the per-origin fit

For every training example, the label is its historical quarterly change. Its inputs use only quarters preceding that example. The forecast target's outcome is never joined into training.

StandardScaler and Ridge are fitted afresh at each origin using only that LGA's earlier rows. We check the training means, variances and sample counts, independently solve the regularised least-squares problem, then verify both the raw change and the final half-adjusted forecast.

The raw predicted change is retained as **fit evidence**, not scored as an extra candidate. Changing or deleting future outcomes must not change an already-issued prediction.

In [3]:
def combine_forecast(last_rent, raw_change, weight):
    require(np.isfinite([last_rent, raw_change, weight]).all() and 0 <= weight <= 1,
            'Invalid anchor, change or damping weight.')
    return float(last_rent + weight*raw_change)


def make_lag_features(frame):
    """Create predictor-only rows. No target-quarter rent or count is returned."""
    require(set(MODEL_COLUMNS).issubset(frame.columns), 'Missing history columns.')
    require(not frame.duplicated(SERIES_KEYS + ['quarter']).any(), 'Duplicate rental key.')
    require(frame[SERIES_KEYS].notna().all().all(), 'Missing series identifiers.')
    pieces = []
    for _, group in frame.groupby(SERIES_KEYS, sort=True, observed=True):
        g = group.copy()
        g['_period'] = pd.PeriodIndex(g['quarter'], freq='Q-DEC')
        g = g.sort_values('_period', kind='stable').reset_index(drop=True)
        idx = pd.PeriodIndex(g['_period'], freq='Q-DEC')
        require(len(g) > 4 and idx.equals(pd.period_range(idx.min(), idx.max(), freq='Q-DEC')),
                'Gap in history or too few quarters for four lags.')
        values = g['median_weekly_rent'].to_numpy(dtype=float)
        require(np.isfinite(values).all() and (values > 0).all(), 'Nonpositive or nonfinite rental input.')
        require(g['source_median_cell'].astype('string').str.fullmatch(r'[A-Z]+[1-9][0-9]*').all(),
                'Missing/malformed source cell.')
        f = g[SERIES_KEYS + ['source_file','source_sheet']].copy()
        f['target_quarter'] = idx.astype(str)
        f['target_index'] = idx.asi8
        f['forecast_origin_quarter'] = (idx-1).astype(str)
        for lag in LAGS:
            f[f'rent_lag_{lag}'] = g['median_weekly_rent'].astype(float).shift(lag)
            f[f'lag_{lag}_quarter'] = g['quarter'].shift(lag)
            f[f'lag_{lag}_source_cell'] = g['source_median_cell'].shift(lag)
        for q in [2,3,4]:
            f[f'target_is_q{q}'] = (idx.quarter == q).astype(float)
        f = f.iloc[4:].copy()  # Exactly four warm-up targets; never drop missing rows silently.
        require(f[RAW_FEATURES].notna().all().all(), 'Missing predictor after warm-up.')
        for k in [1,2,3]:
            f[f'change_lag_{k}'] = f[f'rent_lag_{k}'] - f[f'rent_lag_{k+1}']
        pieces.append(f)
    require(bool(pieces), 'No rental series available.')
    return pd.concat(pieces, ignore_index=True).sort_values(TARGET_KEYS, kind='stable').reset_index(drop=True)


def outcome_table(frame):
    result = frame[SERIES_KEYS+['quarter','median_weekly_rent','source_median_cell']].rename(columns={
        'quarter':'target_quarter','median_weekly_rent':'actual_weekly_rent_aud',
        'source_median_cell':'target_source_median_cell'})
    require(not result.duplicated(TARGET_KEYS).any(), 'Duplicate target outcome.')
    return result.copy()


def fit_one_origin(feature_frame, outcomes, area, property_type, target, minimum_examples=40):
    """Fit a fresh preprocessing/model pipeline using this LGA's earlier targets only."""
    target = pd.Period(target, freq='Q-DEC')
    f = feature_frame.loc[feature_frame.rental_area_name.eq(area)
                          & feature_frame.property_type.eq(property_type)].copy()
    require(not f.duplicated(TARGET_KEYS).any(), 'Duplicate feature key.')
    training = f.loc[f.target_index.lt(target.ordinal)].copy()
    predict_row = f.loc[f.target_index.eq(target.ordinal)].copy()
    require(len(predict_row) == 1, 'Exactly one forecast target is required.')
    require(len(training) >= minimum_examples, 'Insufficient training examples.')
    require(training.target_index.max() == target.ordinal-1, 'Training must end at the preceding quarter.')
    # Target/future outcome rows are removed BEFORE their values are joined or fitted.
    label_period = pd.PeriodIndex(outcomes.target_quarter, freq='Q-DEC')
    labels = outcomes.loc[outcomes.rental_area_name.eq(area) & outcomes.property_type.eq(property_type)
                          & (label_period < target)].copy()
    require(not labels.duplicated(TARGET_KEYS).any(), 'Duplicate training outcome.')
    training = training.merge(labels, on=TARGET_KEYS, how='left', validate='one_to_one', indicator=True)
    require(training['_merge'].eq('both').all(), 'A training outcome is missing.')
    training = training.sort_values('target_index', kind='stable')
    require(training[FEATURES+['actual_weekly_rent_aud']].notna().all().all(), 'Missing training value.')
    X = training[FEATURES].astype(float)
    y = (training.actual_weekly_rent_aud - training.rent_lag_1).to_numpy(dtype=float)
    X_target = predict_row[FEATURES].astype(float)
    require(np.isfinite(X.to_numpy()).all() and np.isfinite(y).all() and
            np.isfinite(X_target.to_numpy()).all(), 'Nonfinite model input.')
    pipeline = Pipeline([('scale', StandardScaler()),
                         ('ridge', Ridge(alpha=1.0, fit_intercept=True, solver='svd'))])
    pipeline.fit(X, y)
    raw_change = float(pipeline.predict(X_target)[0])
    anchor = float(predict_row.iloc[0].rent_lag_1)
    forecast = combine_forecast(anchor, raw_change, DAMPING)
    scaler = pipeline.named_steps['scale']; estimator = pipeline.named_steps['ridge']
    require(int(scaler.n_samples_seen_) == len(X), 'Scaler sample count differs from training rows.')
    np.testing.assert_allclose(scaler.mean_, X.to_numpy().mean(axis=0), rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(scaler.var_, X.to_numpy().var(axis=0), rtol=1e-12, atol=1e-12)

    # Independent closed-form Ridge calculation checks both the fit and prediction.
    z = (X.to_numpy()-scaler.mean_) / scaler.scale_
    z_mean = z.mean(axis=0)
    z_centered = z-z_mean
    y_mean = y.mean()
    coefficient_check = np.linalg.solve(z_centered.T @ z_centered + np.eye(len(FEATURES)),
                                       z_centered.T @ (y-y_mean))
    target_z = (X_target.to_numpy()[0]-scaler.mean_) / scaler.scale_
    independent = float((target_z-z_mean) @ coefficient_check+y_mean)
    np.testing.assert_allclose(raw_change, independent, rtol=1e-9, atol=1e-8)
    np.testing.assert_allclose(forecast, anchor + DAMPING*independent, rtol=1e-9, atol=1e-8)
    require(np.isfinite(forecast), 'Nonfinite Ridge forecast.')
    raw_coefs = estimator.coef_ / scaler.scale_
    raw_intercept = float(estimator.intercept_ - raw_coefs @ scaler.mean_)
    np.testing.assert_allclose(raw_change, raw_intercept+X_target.to_numpy()[0]@raw_coefs,
                               rtol=1e-10, atol=1e-8)
    base = {**dict(zip(SERIES_KEYS, [area, property_type])), 'target_quarter':str(target),
            'forecast_origin_quarter':str(target-1)}
    prediction = {**base,'model':MODEL_NAME,'forecast_weekly_rent_aud':forecast,
                  'last_observed_rent_aud_week':anchor, 'raw_predicted_change_aud_week':raw_change,
                  'applied_adjustment_aud_week':DAMPING*raw_change,'damping_weight':DAMPING}
    audit = {**base, 'training_first_target':str(training.target_quarter.min()),
             'training_last_target':str(training.target_quarter.max()), 'training_rows':len(training),
             'alpha':1.0, 'scaler_training_rows':int(scaler.n_samples_seen_),
             'raw_intercept':raw_intercept, 'training_mean_change_aud_week':float(y_mean),
             'independent_raw_change_aud_week':independent,
             'independent_forecast_aud_week':anchor+DAMPING*independent,'damping_weight':DAMPING,
             'training_targets_at_or_after_forecast_target':0}
    coefficients = [{**base,'feature':name,'training_mean':float(scaler.mean_[j]),
                     'training_scale':float(scaler.scale_[j]),'coefficient_standardized':float(estimator.coef_[j]),
                     'coefficient_raw_units':float(raw_coefs[j])} for j,name in enumerate(FEATURES)]
    p = predict_row.iloc[0]
    lineage = []
    for lag in LAGS:
        require(pd.Period(p[f'lag_{lag}_quarter'],freq='Q-DEC') == target-lag,
                'Feature lineage is not the correct historical quarter.')
        lineage.append({**base,'feature':f'rent_lag_{lag}','lag_quarters':lag,
                        'source_quarter':p[f'lag_{lag}_quarter'],'source_value_aud_week':float(p[f'rent_lag_{lag}']),
                        'source_file':p.source_file,'source_sheet':p.source_sheet,
                        'source_median_cell':p[f'lag_{lag}_source_cell']})
    return prediction,audit,coefficients,lineage


def metric_table(frame, groups):
    require(len(frame) > 0 and np.isfinite(frame.error_aud_week).all(), 'Metrics require finite errors.')
    rows=[]
    by=groups[0] if len(groups)==1 else groups
    for key,g in frame.groupby(by, sort=True, observed=True):
        keys=(key,) if len(groups)==1 else key
        e=g.error_aud_week.to_numpy(dtype=float)
        rows.append({**dict(zip(groups,keys)), 'n_forecasts':len(e),
                     'mae_aud_week':float(np.mean(np.abs(e))),
                     'rmse_aud_week':float(np.sqrt(np.mean(e*e))),
                     'mean_signed_error_aud_week':float(e.mean())})
    return pd.DataFrame(rows)

print('Change-feature builder and baseline-centred per-origin fit defined.')
print('No future/current-quarter targets are passed to model.fit().')


Change-feature builder and baseline-centred per-origin fit defined.
No future/current-quarter targets are passed to model.fit().


## 4. Synthetic self-checks before reading candidate scores

These small fictional series test lag construction, missing/duplicate quarters, separate LGAs, chronology, preprocessing and the independent solution. Additional checks verify that constant histories predict no change, shifting every rent by a constant shifts the forecast equally but leaves the adjustment unchanged, and the damping arithmetic has the intended endpoints.

Alternative weights are used only in a three-number arithmetic check, **not** in official-data model search. These checks do not add pytest cases to the project test suite.

In [4]:
# These fixtures contain no official data and do not write files.
def toy_series(name='Fixture A', shift=0.0):
    p = pd.period_range('2005Q1','2011Q4',freq='Q-DEC')
    vals = [200.0 + 3*i + (i%4)*2 + shift for i in range(len(p))]
    return pd.DataFrame({'rental_area_name':name,'property_type':'2 bedroom flats',
                         'quarter':p.astype(str),'median_weekly_rent':vals,
                         'source_file':'fictional.xlsx','source_sheet':'fixture',
                         'source_median_cell':[f'A{i+1}' for i in range(len(p))]})

def expect_error(call, phrase):
    try: call()
    except ValueError as exc:
        require(phrase in str(exc), f'Unexpected rejection: {exc}')
    else: raise AssertionError(f'Expected rejection: {phrase}')

self_checks=[]
toy=toy_series(); toy_f=make_lag_features(toy); toy_y=outcome_table(toy)
first=toy_f.iloc[0]
require(first.target_quarter=='2006Q1' and first.rent_lag_1==215 and first.rent_lag_4==200,
        'Wrong toy lag values.')
require([first.change_lag_1,first.change_lag_2,first.change_lag_3] == [5.0,5.0,5.0], 'Wrong lagged changes.')
require(first.lag_1_source_cell=='A4' and first.lag_4_source_cell=='A1','Wrong toy source cells.')
self_checks.append('known historical values, quarter labels and source cells')
pd.testing.assert_frame_equal(toy_f,make_lag_features(toy.sample(frac=1,random_state=21)))
self_checks.append('row-order independence')
expect_error(lambda:make_lag_features(pd.concat([toy,toy.iloc[[0]]],ignore_index=True)),'Duplicate')
self_checks.append('duplicate-quarter rejection')
expect_error(lambda:make_lag_features(toy.drop(index=9)),'Gap')
self_checks.append('missing-quarter rejection')
bad=toy.copy();bad.loc[4,'median_weekly_rent']=np.nan
expect_error(lambda:make_lag_features(bad),'Nonpositive or nonfinite')
self_checks.append('missing-value rejection without filling')
combined_toy=pd.concat([toy,toy_series('Fixture B',700)],ignore_index=True)
combined_f=make_lag_features(combined_toy)
require(combined_f.loc[combined_f.rental_area_name.eq('Fixture B')].iloc[0].rent_lag_4==900,
        'Features mixed LGA histories.')
self_checks.append('lags remain inside each LGA')
for q in [1,2,3,4]:
    r=toy_f.loc[pd.PeriodIndex(toy_f.target_quarter,freq='Q-DEC').quarter==q].iloc[0]
    require([r[f'target_is_q{k}'] for k in [2,3,4]]==[float(q==k) for k in [2,3,4]],'Bad calendar encoding.')
self_checks.append('target quarter is known calendar information')
T='2010Q1'
result=fit_one_origin(toy_f,toy_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
require(result[1]['training_last_target']=='2009Q4','Bad forecast origin.')
self_checks.append('training origin and independent Ridge solution')
require(result[1]['training_rows']==16 and result[1]['scaler_training_rows']==16,'Scaler saw target rows.')
self_checks.append('preprocessing fitted only on training examples')

# Mutating current/future actual rents must not change the forecast issued at T.
mut=toy.copy();future=pd.PeriodIndex(mut.quarter,freq='Q-DEC')>=pd.Period(T,freq='Q-DEC')
mut.loc[future,'median_weekly_rent']+=100000
mut_f=make_lag_features(mut); mut_y=outcome_table(mut)
after=fit_one_origin(mut_f,mut_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],after[0]['forecast_weekly_rent_aud'],rtol=0,atol=1e-10)
self_checks.append('current and future target mutation cannot change an issued forecast')

# Completely remove current/future outcomes; feature values remain available from past history.
past_y=toy_y.loc[pd.PeriodIndex(toy_y.target_quarter,freq='Q-DEC')<pd.Period(T,freq='Q-DEC')]
without=fit_one_origin(toy_f,past_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],without[0]['forecast_weekly_rent_aud'],atol=1e-10)
self_checks.append('prediction succeeds without current/future outcomes')

isolated=fit_one_origin(combined_f,outcome_table(combined_toy),'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],isolated[0]['forecast_weekly_rent_aud'],atol=1e-10)
self_checks.append('other LGA outcomes cannot affect the separate model')
expect_error(lambda:fit_one_origin(toy_f,toy_y,'Fixture A','2 bedroom flats','2006Q1',minimum_examples=8),
             'Insufficient')
self_checks.append('insufficient-training rejection')
# Final-period values are discarded by the same development-only selector used above.
toy_full=pd.concat([toy, toy.assign(quarter=pd.period_range('2024Q1','2030Q4',freq='Q-DEC').astype(str))],ignore_index=True)
a=development_only(toy_full)
toy_full.loc[pd.PeriodIndex(toy_full.quarter,freq='Q-DEC')>=HOLDOUT[0],'median_weekly_rent']=999999
pd.testing.assert_frame_equal(a,development_only(toy_full))
self_checks.append('reserved/future outcomes excluded before feature construction')

flat = toy.copy(); flat['median_weekly_rent'] = 500.0
flat_result = fit_one_origin(make_lag_features(flat), outcome_table(flat), 'Fixture A', '2 bedroom flats', T, 8)
np.testing.assert_allclose(flat_result[0]['forecast_weekly_rent_aud'], 500.0, rtol=0, atol=1e-10)
np.testing.assert_allclose(flat_result[0]['raw_predicted_change_aud_week'], 0.0, rtol=0, atol=1e-10)
self_checks.append('constant history produces zero learned correction')
shifted = toy.copy(); shifted['median_weekly_rent'] += 1000.0
translated = fit_one_origin(make_lag_features(shifted), outcome_table(shifted), 'Fixture A', '2 bedroom flats', T, 8)
np.testing.assert_allclose(translated[0]['forecast_weekly_rent_aud'], result[0]['forecast_weekly_rent_aud']+1000, atol=1e-10)
np.testing.assert_allclose(translated[0]['raw_predicted_change_aud_week'], result[0]['raw_predicted_change_aud_week'], atol=1e-10)
self_checks.append('uniform level translation leaves the learned change unchanged')
require(combine_forecast(500, 20, 0)==500 and combine_forecast(500,20,1)==520
        and combine_forecast(500,20,0.5)==510 and combine_forecast(500,-20,0.5)==490, 'Damping arithmetic failed.')
expect_error(lambda:combine_forecast(500,20,1.5), 'Invalid')
self_checks.append('half adjustment, zero/full endpoints and invalid weight rejection')
missing_outcome = toy_y.loc[toy_y.target_quarter.ne('2009Q4')].copy()
expect_error(lambda:fit_one_origin(toy_f,missing_outcome,'Fixture A','2 bedroom flats',T,8), 'training outcome')
self_checks.append('missing earlier training outcome stops instead of silently dropping a target')
require(len(self_checks)==18, 'Unexpected self-check count.')
print(f'{len(self_checks)} correction self-check groups passed.')
print('No official-data parameters were searched; the final period remains excluded.')


18 correction self-check groups passed.
No official-data parameters were searched; the final period remains excluded.


## 5. Build features and trace each input to its source

Four initial warm-up quarters are omitted from each series. The remaining 2,914 predictor rows stop at 2023Q3. All 11,656 underlying lag values, source quarters and workbook cells are checked through direct key lookups, separately from the shift operation. The three change features are independently reconstructed from those values.

We also match the saved baseline forecasts and all earlier scored actuals to this same unchanged rental history.

In [5]:
features = make_lag_features(development)
require(len(features)==2914, 'Expected 31 x 94 supervised predictor rows (four warm-up quarters removed).')
require(features.target_quarter.min()=='2000Q2' and features.target_quarter.max()=='2023Q3',
        'Unexpected supervised feature period.')
require('median_weekly_rent' not in features and 'published_count' not in features,
        'Target or count leaked into predictors.')
require(pd.PeriodIndex(features.target_quarter,freq='Q-DEC').max() < HOLDOUT[0],
        'Final-period feature rows must not be constructed.')

# Check every lag via an independent exact-key lookup, not shift() or model.fit().
truth=development.set_index(SERIES_KEYS+['quarter'])
lag_entries_checked=0
for row in features.itertuples(index=False):
    target=pd.Period(row.target_quarter,freq='Q-DEC')
    for lag in LAGS:
        original=truth.loc[(row.rental_area_name,row.property_type,str(target-lag))]
        require(float(getattr(row,f'rent_lag_{lag}'))==float(original.median_weekly_rent),
                'Lag value differs from independent source lookup.')
        require(getattr(row,f'lag_{lag}_quarter')==str(target-lag)
                and getattr(row,f'lag_{lag}_source_cell')==original.source_median_cell,
                'Lag quarter/cell differs from source lookup.')
        lag_entries_checked+=1
require(lag_entries_checked==11656,'Lag-check count changed.')

# Recheck the saved baseline predictions and actuals against the unchanged history.
for row in baseline_predictions.itertuples(index=False):
    target=pd.Period(row.target_quarter,freq='Q-DEC')
    lag=1 if row.model=='last_observation' else 4
    expected=truth.loc[(row.rental_area_name,row.property_type,str(target-lag))]
    actual=truth.loc[(row.rental_area_name,row.property_type,str(target))]
    require(float(row.forecast_weekly_rent_aud)==float(expected.median_weekly_rent)
            and row.forecast_source_quarter==str(target-lag)
            and row.forecast_source_median_cell==expected.source_median_cell,
            'Saved baseline forecast differs from its historical source.')
    require(float(row.actual_weekly_rent_aud)==float(actual.median_weekly_rent)
            and row.target_source_median_cell==actual.source_median_cell,
            'Saved baseline actual differs from the input rental table.')

print('FEATURE CHECKPOINT PASSED')
print('Supervised predictor rows: 2914 | Columns used by correction model: 6')
print('Independent historical value/quarter/cell matches:',lag_entries_checked)
print('Previous baseline prediction rows verified: 1178')
display(features.loc[features.rental_area_name.eq('Melbourne') &
                     features.target_quarter.isin(['2019Q1','2019Q2']),
                    SERIES_KEYS+['target_quarter','forecast_origin_quarter',*FEATURES]])

change_entries_checked = 0
for r in features.itertuples(index=False):
    t=pd.Period(r.target_quarter, freq='Q-DEC')
    for k in [1,2,3]:
        a=truth.loc[(r.rental_area_name,r.property_type,str(t-k))].median_weekly_rent
        b=truth.loc[(r.rental_area_name,r.property_type,str(t-k-1))].median_weekly_rent
        np.testing.assert_allclose(getattr(r,f'change_lag_{k}'), float(a)-float(b), rtol=0,atol=0)
        change_entries_checked += 1
require(change_entries_checked == 8742, 'Unexpected lag-change checks.')
for r in old_predictions.itertuples(index=False):
    actual=truth.loc[(r.rental_area_name,r.property_type,r.target_quarter)]
    require(float(r.actual_weekly_rent_aud)==float(actual.median_weekly_rent)
            and r.target_source_median_cell==actual.source_median_cell, 'Earlier actual/source reference changed.')
print('Independent lag-change values matched:',change_entries_checked)
print('Earlier forecast actual/source pairs verified:',len(old_predictions))


FEATURE CHECKPOINT PASSED
Supervised predictor rows: 2914 | Columns used by correction model: 6
Independent historical value/quarter/cell matches: 11656
Previous baseline prediction rows verified: 1178


,rental_area_name,property_type,target_quarter,forecast_origin_quarter,change_lag_1,change_lag_2,change_lag_3,target_is_q2,target_is_q3,target_is_q4
1673,Melbourne,2 bedroom flats,2019Q1,2018Q4,-13.0,13.0,0.0,0.0,0.0,0.0
1674,Melbourne,2 bedroom flats,2019Q2,2019Q1,20.0,-13.0,13.0,1.0,0.0,0.0


Independent lag-change values matched: 8742
Earlier forecast actual/source pairs verified: 1767


## 6. Fit one candidate at each validation origin

There are 31 × 19 = 589 separate fits. The training windows contain 75–93 earlier supervised examples per fit. Each target outcome is attached only after its prediction has been made. Progress is printed once per LGA; do not repeatedly press Run while the cell is working.

In [6]:
# 19 expanding-window fits per LGA. No alpha search and no holdout fits.
outcomes=outcome_table(development)
rows=[]; audit_rows=[]; coefficient_rows=[]; lineage_rows=[]
for i,area in enumerate(LGAS,start=1):
    area_features=features.loc[features.rental_area_name.eq(area)].copy()
    area_outcomes=outcomes.loc[outcomes.rental_area_name.eq(area)].copy()
    for target in VALIDATION:
        pred, audit, coefs, lineage=fit_one_origin(
            area_features, area_outcomes, area, protocol['property_type'], target,
            minimum_examples=correction_plan['minimum_training_examples'])
        rows.append(pred); audit_rows.append(audit)
        coefficient_rows.extend(coefs); lineage_rows.extend(lineage)
    print(f'{i:02d}/31 {area}: 19 validation forecasts completed.',flush=True)
correction_predictions=pd.DataFrame(rows).sort_values(TARGET_KEYS,kind='stable').reset_index(drop=True)
fit_audit=pd.DataFrame(audit_rows).sort_values(TARGET_KEYS,kind='stable').reset_index(drop=True)
coefficient_audit=pd.DataFrame(coefficient_rows).sort_values([*TARGET_KEYS,'feature'],kind='stable').reset_index(drop=True)
feature_lineage=pd.DataFrame(lineage_rows).sort_values([*TARGET_KEYS,'lag_quarters'],kind='stable').reset_index(drop=True)
require(len(correction_predictions)==589 and not correction_predictions.duplicated(TARGET_KEYS).any(),
        'Unexpected correction forecast keys/count.')
require(correction_predictions.target_quarter.isin(VALIDATION.astype(str)).all(),'Ridge targets outside validation.')
require(len(fit_audit)==589 and len(coefficient_audit)==3534 and len(feature_lineage)==2356,
        'Fit evidence counts changed.')
require(fit_audit.training_rows.min()==75 and fit_audit.training_rows.max()==93,
        'Unexpected expanding-training sizes.')
require((pd.PeriodIndex(fit_audit.training_last_target,freq='Q-DEC') ==
         pd.PeriodIndex(fit_audit.target_quarter,freq='Q-DEC')-1).all(), 'Model trained after its origin.')
print('CORRECTION FORECAST CHECKPOINT PASSED')
print('Fitted LGA/origin models: 589 | Training examples per fit: 75-93')
print('Independent closed-form predictions matched: 589')
print('Forecast lag-source records: 2356 | Final-period forecasts: 0')


01/31 Banyule: 19 validation forecasts completed.
02/31 Bayside: 19 validation forecasts completed.
03/31 Boroondara: 19 validation forecasts completed.
04/31 Brimbank: 19 validation forecasts completed.
05/31 Cardinia: 19 validation forecasts completed.
06/31 Casey: 19 validation forecasts completed.
07/31 Darebin: 19 validation forecasts completed.
08/31 Frankston: 19 validation forecasts completed.
09/31 Glen Eira: 19 validation forecasts completed.
10/31 Greater Dandenong: 19 validation forecasts completed.
11/31 Hobsons Bay: 19 validation forecasts completed.
12/31 Hume: 19 validation forecasts completed.
13/31 Kingston: 19 validation forecasts completed.
14/31 Knox: 19 validation forecasts completed.
15/31 Manningham: 19 validation forecasts completed.
16/31 Maribyrnong: 19 validation forecasts completed.
17/31 Maroondah: 19 validation forecasts completed.
18/31 Melbourne: 19 validation forecasts completed.
19/31 Melton: 19 validation forecasts completed.
20/31 Merri-bek: 19 vali

## 7. Score the candidate against the three saved methods

Every method is compared on the same 589 targets. We retain the previous Ridge forecasts exactly as saved and recalculate its earlier metrics to ensure they have not changed. No old method is refitted here.

MAE is the primary measure. Equal weight is given to every LGA-quarter; RMSE is pooled from individual squared errors. Signed error is forecast minus actual. There is no automatic selection of a different model for each LGA, no confidence interval or significance claim, and no deployment decision.

In [7]:
CORE = ['model',*TARGET_KEYS,'forecast_origin_quarter','forecast_weekly_rent_aud']
predictions = pd.concat([old_predictions[CORE], correction_predictions[CORE]],ignore_index=True)
require(not predictions.duplicated(['model',*TARGET_KEYS]).any(),'Duplicate prediction key.')
comparison=predictions.merge(outcomes,on=TARGET_KEYS,how='left',validate='many_to_one',indicator=True)
require(comparison['_merge'].eq('both').all(),'Missing scored outcome.')
comparison=comparison.drop(columns='_merge').sort_values(['model',*TARGET_KEYS],kind='stable').reset_index(drop=True)
comparison['error_aud_week']=comparison.forecast_weekly_rent_aud-comparison.actual_weekly_rent_aud
comparison['absolute_error_aud_week']=comparison.error_aud_week.abs()
comparison['squared_error']=comparison.error_aud_week.pow(2)
require(len(comparison)==2356 and comparison.groupby('model').size().eq(589).all(), 'Unequal forecast coverage.')
require(comparison.target_quarter.isin(VALIDATION.astype(str)).all(), 'Final period was accidentally scored.')
correction_scored=comparison.loc[comparison.model.eq(MODEL_NAME)].reset_index(drop=True)
correction_scored=correction_scored.merge(correction_predictions[TARGET_KEYS+[
    'last_observed_rent_aud_week','raw_predicted_change_aud_week','applied_adjustment_aud_week','damping_weight']],
    on=TARGET_KEYS,how='left',validate='one_to_one')
np.testing.assert_allclose(correction_scored.forecast_weekly_rent_aud,
    correction_scored.last_observed_rent_aud_week+correction_scored.applied_adjustment_aud_week,rtol=1e-12,atol=1e-10)
overall_metrics=metric_table(comparison,['model'])
lga_metrics=metric_table(comparison,['model',*SERIES_KEYS])
quarter_metrics=metric_table(comparison,['model','target_quarter'])
METRIC_COLUMNS=['n_forecasts','mae_aud_week','rmse_aud_week','mean_signed_error_aud_week']
for name,new_table,keys in [
    ('validation_metrics_overall.csv',overall_metrics,['model']),
    ('validation_metrics_by_lga.csv',lga_metrics,['model',*SERIES_KEYS]),
    ('validation_metrics_by_quarter.csv',quarter_metrics,['model','target_quarter'])]:
    before=old_tables[name].sort_values(keys).reset_index(drop=True)
    after=new_table.loc[new_table.model.ne(MODEL_NAME)].sort_values(keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(before[keys].astype('string'),after[keys].astype('string'),check_dtype=False)
    np.testing.assert_allclose(before[METRIC_COLUMNS].to_numpy(float),after[METRIC_COLUMNS].to_numpy(float),rtol=1e-10,atol=1e-9)
old_errors=old_predictions.sort_values(['model',*TARGET_KEYS]).reset_index(drop=True)
retained=comparison.loc[comparison.model.ne(MODEL_NAME)].reset_index(drop=True)
np.testing.assert_allclose(old_errors[['forecast_weekly_rent_aud','error_aud_week','squared_error']],
    retained[['forecast_weekly_rent_aud','error_aud_week','squared_error']],rtol=1e-10,atol=1e-8)
require(len(lga_metrics)==124 and lga_metrics.n_forecasts.eq(19).all(), 'LGA coverage changed.')
require(len(quarter_metrics)==76 and quarter_metrics.n_forecasts.eq(31).all(), 'Quarter coverage changed.')
for m in overall_metrics.model:
    pooled=overall_metrics.loc[overall_metrics.model.eq(m)].iloc[0]
    g=lga_metrics.loc[lga_metrics.model.eq(m)]
    np.testing.assert_allclose(pooled.mae_aud_week,g.mae_aud_week.mean(),rtol=1e-12,atol=1e-12)
    np.testing.assert_allclose(pooled.rmse_aud_week,np.sqrt(np.mean(g.rmse_aud_week**2)),rtol=1e-12,atol=1e-12)
paired=lga_metrics.pivot(index=SERIES_KEYS,columns='model',values='mae_aud_week').reset_index()
paired.columns.name=None
paired=paired.rename(columns={'last_observation':'last_observation_mae',MODEL_NAME:'correction_mae',
                              OLD_RIDGE_NAME:'previous_level_ridge_mae','seasonal_naive':'seasonal_naive_mae'})
paired['correction_minus_last_mae_aud_week']=paired.correction_mae-paired.last_observation_mae
paired['correction_vs_last']=np.select([paired.correction_minus_last_mae_aud_week.lt(-1e-9),
    paired.correction_minus_last_mae_aud_week.gt(1e-9)],['correction_lower_mae','last_lower_mae'],default='numerical_tie')
last_mae=float(overall_metrics.set_index('model').loc['last_observation','mae_aud_week'])
new_mae=float(overall_metrics.set_index('model').loc[MODEL_NAME,'mae_aud_week'])
mae_change_pct=100*(new_mae/last_mae-1)
MODEL_LABELS={'last_observation':'Last observation','seasonal_naive':'Seasonal naive',
    OLD_RIDGE_NAME:'Earlier level Ridge',MODEL_NAME:'Half-adjusted change Ridge'}
MODEL_ORDER=['last_observation','seasonal_naive',OLD_RIDGE_NAME,MODEL_NAME]
print('FOUR-METHOD VALIDATION CHECKPOINT PASSED')
display(overall_metrics.round(4))
print(f'Correction-model MAE change versus last observation: {mae_change_pct:+.2f}% (negative is better).')
print('Validation has already informed the design. This is not fresh independent evidence.')
display(paired.correction_vs_last.value_counts().rename_axis('comparison').reset_index(name='lga_count'))


FOUR-METHOD VALIDATION CHECKPOINT PASSED


,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,damped_change_ridge_alpha1_weight05,589,10.9133,15.4055,-2.7078
1,last_observation,589,10.9728,16.2303,-4.3582
2,ridge_lags_seasonal_alpha1,589,13.2135,18.6421,-1.9758
3,seasonal_naive,589,25.2937,38.9464,-13.0934


Correction-model MAE change versus last observation: -0.54% (negative is better).
Validation has already informed the design. This is not fresh independent evidence.


,comparison,lga_count
0,correction_lower_mae,16
1,last_lower_mae,15


## 8. Charts: overall error and error over time

The same targets are used by all four methods. Lower MAE is better. The quarter-by-quarter chart can show unstable improvements that a single overall number would hide. No quarter is removed because its error is large.

In [8]:
chart_metrics=overall_metrics.copy();chart_metrics['method']=chart_metrics.model.map(MODEL_LABELS)
figures={}
fig=px.bar(chart_metrics,x='method',y='mae_aud_week',text='mae_aud_week',
    category_orders={'method':[MODEL_LABELS[m] for m in MODEL_ORDER]},
    labels={'method':'Method','mae_aud_week':'Validation MAE (AUD/week)'},
    title='One fixed correction candidate versus saved benchmarks',
    hover_data={'n_forecasts':True,'rmse_aud_week':':.4f','mean_signed_error_aud_week':':.4f'})
fig.update_traces(texttemplate='%{text:.2f}',textposition='outside')
fig.update_layout(height=560,margin=dict(b=125,t=90),xaxis_tickangle=-15)
figures['01_validation_mae.html']=fig;fig.show()
qchart=quarter_metrics.copy();qchart['method']=qchart.model.map(MODEL_LABELS)
qchart=qchart.sort_values(['method','target_quarter'])
fig=px.line(qchart,x='target_quarter',y='mae_aud_week',color='method',markers=True,
    labels={'target_quarter':'Validation target quarter','mae_aud_week':'MAE (AUD/week)','method':'Method'},
    title='Validation error by quarter — 31 equally weighted LGAs per point')
fig.update_layout(height=600,legend=dict(orientation='h',y=-0.3),margin=dict(b=145),xaxis_tickangle=-45)
figures['02_validation_mae_by_quarter.html']=fig;fig.show()


## 9. Chart: per-LGA change in MAE relative to last observation

A negative bar favours the new candidate; a positive bar favours last observation. These are descriptive validation comparisons, not permission to pick area-specific winners from the same evaluated outcomes.

In [9]:
lgchart=paired.sort_values(['correction_minus_last_mae_aud_week','rental_area_name'])
fig=px.bar(lgchart,x='correction_minus_last_mae_aud_week',y='rental_area_name',orientation='h',
    labels={'rental_area_name':'LGA','correction_minus_last_mae_aud_week':'Correction MAE minus last-observation MAE (AUD/week)'},
    hover_data={'correction_mae':':.4f','last_observation_mae':':.4f'},
    title='Per-LGA validation gap — left favours correction, right favours last observation')
fig.add_vline(x=0,line_dash='dash')
fig.update_layout(height=1050,margin=dict(l=175,t=90,b=90),yaxis=dict(categoryorder='array',categoryarray=lgchart.rental_area_name.tolist()))
figures['03_correction_gap_by_lga.html']=fig;fig.show()


## 10. Chart: Melbourne's actual rents and validation forecasts

Melbourne is the same illustrative LGA used earlier, not a favourable case selected after viewing these scores. Only the validation dates appear. Hover over a forecast for its origin, actual and error. The plot contains no reserved final-period results and no new forecast beyond the source history.

In [10]:
EXAMPLE_LGA='Melbourne'
example=comparison.loc[comparison.rental_area_name.eq(EXAMPLE_LGA)].copy()
actual=example.loc[example.model.eq('last_observation')].sort_values('target_quarter')
fig=go.Figure()
fig.add_trace(go.Scatter(x=actual.target_quarter,y=actual.actual_weekly_rent_aud,name='Published actual',mode='lines+markers'))
for m in MODEL_ORDER:
    part=example.loc[example.model.eq(m)].sort_values('target_quarter')
    extra=part[['forecast_origin_quarter','actual_weekly_rent_aud','error_aud_week']].to_numpy()
    fig.add_trace(go.Scatter(x=part.target_quarter,y=part.forecast_weekly_rent_aud,name=MODEL_LABELS[m],
        mode='lines+markers',customdata=extra,
        hovertemplate='Target %{x}<br>Forecast $%{y:.2f}/week<br>Origin %{customdata[0]}<br>Actual $%{customdata[1]:.2f}<br>Error $%{customdata[2]:.2f}<extra>%{fullData.name}</extra>'))
fig.update_layout(title='Melbourne — validation forecasts, not current-rent estimates',
    xaxis_title='Validation target quarter',yaxis_title='Published median weekly rent / forecast (AUD)',
    height=680,legend=dict(orientation='h',y=-0.3),margin=dict(b=160),xaxis_tickangle=-45)
figures['04_melbourne_validation_forecasts.html']=fig;fig.show()


## 11. Export the candidate evidence without replacing earlier experiments

Nine CSV tables are staged, reloaded and compared before publication. The prediction evidence retains the anchor, raw predicted change, damping weight and final adjustment. The feature-lineage table preserves the four original quarterly observations underlying each forecast's changes and anchor. The coefficient table records the training means/scales for all six inputs.

Only this notebook's output directory is written. Existing outputs with a different protocol or input lineage are rejected. Individual file replacement is not a multi-file database transaction: do not run two writers simultaneously. Personal notes belong outside the generated report.

In [11]:
exports={
    'correction_validation_predictions.csv':correction_scored,
    'validation_predictions_all_methods.csv':comparison,
    'validation_metrics_overall.csv':overall_metrics,
    'validation_metrics_by_lga.csv':lga_metrics,
    'validation_metrics_by_quarter.csv':quarter_metrics,
    'correction_vs_last_by_lga.csv':paired,
    'fit_audit.csv':fit_audit,
    'preprocessing_and_coefficients.csv':coefficient_audit,
    'forecast_feature_lineage.csv':feature_lineage,
}

def schema_for(frame):
    result={}
    for c,d in frame.dtypes.items():
        if pd.api.types.is_integer_dtype(d): result[c]='int64'
        elif pd.api.types.is_float_dtype(d): result[c]='float64'
        else: result[c]='string'
    return result


def compare_roundtrip(expected,actual):
    require(expected.columns.tolist()==actual.columns.tolist() and len(expected)==len(actual),'CSV schema/row mismatch.')
    for col,kind in schema_for(expected).items():
        if kind=='float64':
            np.testing.assert_allclose(expected[col].to_numpy(dtype=float),actual[col].to_numpy(dtype=float),
                                        rtol=1e-11,atol=1e-10,equal_nan=False)
        elif kind=='int64':
            require(np.array_equal(expected[col].to_numpy(),actual[col].to_numpy()),f'Integer read-back mismatch: {col}')
        else:
            require(expected[col].astype('string').reset_index(drop=True).equals(actual[col].astype('string').reset_index(drop=True)),
                    f'Text read-back mismatch: {col}')

for path,digest in input_hashes.items():
    require(sha256_file(path)==digest,f'Protected input changed: {path.relative_to(ROOT)}')
require(comparison.target_quarter.isin(VALIDATION.astype(str)).all(),'Refusing to export final-period scores.')
OUT.mkdir(parents=True,exist_ok=True)
output_schemas={}
with TemporaryDirectory(prefix='.correction-stage-',dir=OUT) as temporary:
    stage=Path(temporary)
    for filename,frame in exports.items():
        frame.to_csv(stage/filename,index=False,encoding='utf-8',lineterminator='\n')
        schema=schema_for(frame);output_schemas[filename]=schema
        loaded=pd.read_csv(stage/filename,dtype=schema,keep_default_na=False)
        compare_roundtrip(frame,loaded)
    for filename in exports:
        (stage/filename).replace(OUT/filename)
figure_dir=OUT/'figures';figure_dir.mkdir(exist_ok=True)
for filename,figure in figures.items():
    figure.write_html(figure_dir/filename,include_plotlyjs=True,full_html=True,
                      config={'responsive':True,'displaylogo':False})
(OUT/'correction_protocol.json').write_text(json.dumps(correction_plan,indent=2)+'\n',encoding='utf-8')
print('Nine CSV tables reloaded and matched; four interactive charts saved.')
print('No trained model pickle, final-period forecast or deployment data has been created.')


Nine CSV tables reloaded and matched; four interactive charts saved.
No trained model pickle, final-period forecast or deployment data has been created.


## 12. Record the result, including an unfavourable one

A smaller error on this repeatedly examined validation period is a **development result**, not a deployment qualification. The protocol, raw adjustment and all earlier forecasts remain recorded regardless of the outcome. Stop open-ended tweaking after this candidate and review readiness to freeze a model-selection decision before any final-period evaluation.

This remains a single-vintage retrospective study. Historical publication delays and revisions are unknown, historical administrative boundaries are not harmonised here, and medians concern changing sets of lettings rather than a fixed property. Final-period values appeared in earlier EDA, so they are reserved from scoring but not a pristine blind test.

In [12]:
REPORT_PATH=ROOT/'docs/rental_correction_validation.md'
created_at=datetime.now(timezone.utc).isoformat()
counts=paired.correction_vs_last.value_counts()
lines=['# HomeReach Melbourne — Half-adjusted rent-change Ridge validation','',f'Generated (UTC): {created_at}','',
'## Fixed candidate and timing','',
'This candidate was designed after the earlier level-Ridge validation and diagnostics. Its settings were fixed before its own scores were computed, but validation has already informed development; this is not independent evaluation or preregistration.',
'Forecast = previous-quarter rent + 0.5 × predicted quarterly rent change. Separate LGA models use the three preceding quarterly changes and known target Q2/Q3/Q4 indicators.',
'StandardScaler is fitted on the six training-only features at each origin. Ridge alpha=1.0, intercept enabled, SVD solver. Damping=0.5 is a fixed heuristic, not a probability or an optimised value.',
'No grid search, alpha/weight tuning, per-LGA winner selection, observed-direction oracle, clipping or rounding. Raw predicted changes are audit evidence only, not a separately scored candidate.',
'Target transformation, feature transformation and damping change together; their individual causal effects are not isolated.',
'Rolling validation: 2019Q1–2023Q3, 589 equally weighted LGA-quarter targets. Initial history ends 2018Q4; four-lag warm-up yields 75–93 supervised training rows per fit.',
'Final period 2023Q4–2025Q3 is excluded from features, fitting, predictions, metrics and charts. No transport features or current/future forecast.','',
'## Validation results','',
'| Method | Targets | MAE (AUD/week) | RMSE (AUD/week) | Mean signed error (forecast minus actual) |',
'|---|---:|---:|---:|---:|']
for m in MODEL_ORDER:
    r=overall_metrics.set_index('model').loc[m]
    lines.append(f'| {MODEL_LABELS[m]} | {int(r.n_forecasts)} | {r.mae_aud_week:.4f} | {r.rmse_aud_week:.4f} | {r.mean_signed_error_aud_week:.4f} |')
lines += ['',f'Candidate MAE change relative to last observation: {mae_change_pct:+.2f}% (negative is better).',
          f'Lower candidate MAE in {int(counts.get("correction_lower_mae",0))} LGAs; lower last-observation MAE in {int(counts.get("last_lower_mae",0))}; numerical ties: {int(counts.get("numerical_tie",0))}.',
          ('The fixed candidate did not improve pooled validation MAE over last observation.' if new_mae>=last_mae else
           'The fixed candidate improved pooled validation MAE over last observation. This remains inspected-development-set evidence, not final-period or prospective confirmation.'),
          'A smaller signed bias does not imply smaller absolute error. Positive and negative errors can cancel. Error comparisons share areas/time; no iid interval, p-value or significance claim is made.',
          f'Negative candidate rent predictions: {int(correction_scored.forecast_weekly_rent_aud.lt(0).sum())}; no clipping was used.','',
          '## Verification and evidence','',
          f'- {len(self_checks)} fictional self-check groups passed, including future mutation, target-outcome removal, constant histories and translation invariance.',
          '- 11,656 lag values/quarters/source cells and 8,742 lag-change values independently checked.',
          '- 1,178 baseline forecasts checked against the underlying source history; all 1,767 saved predictions retained unchanged.',
          '- 589 candidate fits and raw/final predictions verified against an independent closed-form Ridge solution.',
          '- Training scaler means, variances and sample counts checked per fit. No target/future outcome joins into training.',
          '- 2,356 forecast source-lag records and 3,534 six-feature scaling/coefficient records exported.',
          '- 2,356 scored comparison rows: four methods × the same 589 validation targets.',
          '- Previous overall, per-LGA and per-quarter metrics matched; pooled metrics independently reconciled.',
          '- Nine CSV round trips passed; four validation-only interactive charts saved.',
          '- No previous notebook output, input data, configuration, app, registry or frozen release file overwritten.','',
          '## Research limitations and next gate','',
          'The experiment uses one 2025Q3 workbook vintage. Historical publication lags and revisions are not reconstructed. This is not a real-time backtest.',
          'The reserved period appeared in previous descriptive EDA and the live historical explorer; it is not a pristine unseen test. It remains unscored in the research experiments so far.',
          'The 2019Q1–2023Q3 period has informed multiple modelling decisions. Do not present improvement here as independent confirmation or keep searching until a method wins.',
          'Record this fixed candidate, then review whether to freeze a single selection rule before a separately controlled final-period evaluation. No automatic model promotion or app update occurs.',
          'Published medians are not individual-property prices; nominal changes are not inflation-adjusted. The half correction does not guarantee improvement on flat or declining targets.','',
          '## Reproduction','',
          'Run notebooks/12_rental_correction_validation.ipynb on feat/rental-forecast-baselines after notebooks 09–11.',
          'Research output folder: data/interim/rental_correction_validation/. This generated report is replaced on an identical rerun; keep personal notes elsewhere.',
          f'Protocol SHA-256: {CORRECTION_HASH}.',
          f'Runtime: Python {sys.version.split()[0]}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}.','',
          '## Primary references','',
          '- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html',
          '- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html',
          '- https://scikit-learn.org/stable/common_pitfalls.html',
          '- https://otexts.com/fpp3/tscv.html','']
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True)
REPORT_PATH.write_text('\n'.join(lines),encoding='utf-8')
for path,digest in input_hashes.items():
    require(sha256_file(path)==digest,f'Protected file changed: {path.relative_to(ROOT)}')
manifest={'created_at_utc':created_at,'builder':'notebooks/12_rental_correction_validation.ipynb',
          'research_branch':BRANCH,'source_workbook_sha256':EXPECTED_SOURCE_SHA256,
          'input_csv_sha256':input_hashes[INPUT],'protocol_sha256':CORRECTION_HASH,'protocol':correction_plan,
          'baseline_manifest_sha256':input_hashes[BASE_MANIFEST_PATH],
          'ridge_manifest_sha256':input_hashes[OLD_RIDGE_MANIFEST],
          'diagnostic_manifest_sha256':input_hashes[DIAG_MANIFEST],
          'input_sha256':{str(p.relative_to(ROOT)).replace('\\','/'):h for p,h in input_hashes.items()},
          'versions':{'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,
                      'sklearn':sklearn.__version__,'plotly':plotly.__version__},
          'quality':{'candidate_forecasts':589,'combined_forecasts':2356,'fitted_models':589,
                     'independent_lag_matches':lag_entries_checked,'independent_change_matches':change_entries_checked,
                     'independent_solution_checks':589,'previous_forecast_rows_preserved':1767,
                     'self_check_groups':len(self_checks),'final_period_forecasts':0,'final_period_scores':0,
                     'parameter_combinations_searched':0,'transport_features_used':False,'current_rent_forecast_produced':False,
                     'csv_readback_matches':True,'csv_tables':len(exports),'charts':len(figures),'protected_files_unchanged':True},
          'validation_metrics':overall_metrics.to_dict(orient='records'),'self_checks':self_checks,
          'output_schemas':output_schemas,
          'output_sha256':{str(p.relative_to(OUT)).replace('\\','/'):sha256_file(p)
                           for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='correction_manifest.json'},
          'report_sha256':sha256_file(REPORT_PATH)}
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
print('CORRECTION VALIDATION CHECKPOINT PASSED')
print('Rental LGAs: 31 | Validation quarters: 19 (2019Q1-2023Q3)')
print('New candidate forecasts: 589 | Four-method comparison rows: 2356')
print('Fitted LGA/origin models: 589 | Earlier training examples per fit: 75-93')
print('Fixed damping: 0.5 | Alpha: 1.0 | Parameter combinations searched: 0')
print('Independent historical lag matches: 11656 | Lag-change checks: 8742')
print('Independent raw-change and final-prediction checks: 589')
print('Previous forecast rows preserved: 1767 | Earlier metrics: matched')
print('Final-holdout forecasts scored: 0')
print('Correction self-check groups:',len(self_checks),'passed')
print('CSV tables saved and reloaded:',len(exports),'| Interactive charts:',len(figures))
print('Transport features used: False | Current-rent forecast produced: False')
print('Protected inputs, prior experiments and deployed release unchanged: True')
print('Saved report:',REPORT_PATH.relative_to(ROOT))
print('Saved outputs:',OUT.relative_to(ROOT))
display(overall_metrics.round(4))


CORRECTION VALIDATION CHECKPOINT PASSED
Rental LGAs: 31 | Validation quarters: 19 (2019Q1-2023Q3)
New candidate forecasts: 589 | Four-method comparison rows: 2356
Fitted LGA/origin models: 589 | Earlier training examples per fit: 75-93
Fixed damping: 0.5 | Alpha: 1.0 | Parameter combinations searched: 0
Independent historical lag matches: 11656 | Lag-change checks: 8742
Independent raw-change and final-prediction checks: 589
Previous forecast rows preserved: 1767 | Earlier metrics: matched
Final-holdout forecasts scored: 0
Correction self-check groups: 18 passed
CSV tables saved and reloaded: 9 | Interactive charts: 4
Transport features used: False | Current-rent forecast produced: False
Protected inputs, prior experiments and deployed release unchanged: True
Saved report: docs\rental_correction_validation.md
Saved outputs: data\interim\rental_correction_validation


,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,damped_change_ridge_alpha1_weight05,589,10.9133,15.4055,-2.7078
1,last_observation,589,10.9728,16.2303,-4.3582
2,ridge_lags_seasonal_alpha1,589,13.2135,18.6421,-1.9758
3,seasonal_naive,589,25.2937,38.9464,-13.0934


## Your checkpoint

Save the notebook with **Ctrl+S** and send the final checkpoint with the four-row error table, or the first error. Do not change alpha, damping, evaluation dates or features to chase a better number. Do not commit data/interim outputs or merge research into deployed main at this checkpoint.

A passing check proves that this candidate ran according to its recorded method, **not** that it improved accuracy. A worse score is a valid result. After recording it, we will review model-selection readiness rather than endlessly revisit the same validation outcomes.